In [6]:
import polars as pl


df_raw = pl.read_csv("/Users/rayankhan/Documents/PERSONAL/PROGRAMMING/Python/VoterSuppression/data/2016-precinct-president.csv", infer_schema_length=10000, null_values=["NA", "N/A", ""], encoding="utf8-lossy")

# There are way too many columns in the dataset, these are the only ones I actually need
cols_to_keep = ["year", "state", "state_postal", "state_fips", "county_name", "county_fips", "precinct", "candidate_normalized", "party", "votes", "mode"]


df = df_raw.filter(
   pl.col("party").str.to_lowercase().str.contains("democrat|republican")
)


print(df.shape)
print(df["party"].value_counts().sort("count", descending=True))
print(df["mode"].value_counts().sort("count", descending=True))


# Looks for every column with democratic in it and turns it into democratic otherwise make it republican
df = df.with_columns(
   pl.when(pl.col("party").str.to_lowercase().str.contains("democrat"))
   .then(pl.lit("democrat"))
   .otherwise(pl.lit("republican"))
   .alias("party")
)


df_total = df.filter(pl.col("mode") == "total")
df_nototal = df.filter(pl.col("mode") != "total")


states_with_total = df_total["state"].unique().to_list()


group_cols = [
   "year", "state", "state_postal", "state_fips",
   "county_name", "county_fips", "precinct", "party"
]


df_sum = (
   df_nototal
   .filter(~pl.col("state").is_in(states_with_total))
   .group_by(group_cols)
   .agg(pl.col("votes").sum())
)

# Clean and Recombine
df_cleaned = pl.concat([
   df_total.select(group_cols + ["votes"]),
   df_sum
])

# Viewing Cleaned Dataframe
df_cleaned.head(10)


(470372, 37)
shape: (6, 2)
┌─────────────────────────┬────────┐
│ party                   ┆ count  │
│ ---                     ┆ ---    │
│ str                     ┆ u32    │
╞═════════════════════════╪════════╡
│ republican              ┆ 234807 │
│ democratic              ┆ 224082 │
│ Democratic-Farmer-Labor ┆ 4120   │
│ none / republican       ┆ 3010   │
│ green / democratic      ┆ 3010   │
│ democrat                ┆ 1343   │
└─────────────────────────┴────────┘
shape: (46, 2)
┌───────────────────────────────┬────────┐
│ mode                          ┆ count  │
│ ---                           ┆ ---    │
│ str                           ┆ u32    │
╞═══════════════════════════════╪════════╡
│ total                         ┆ 265859 │
│ election day                  ┆ 91346  │
│ absentee                      ┆ 33158  │
│ Absentee                      ┆ 8418   │
│ provisional                   ┆ 7519   │
│ …                             ┆ …      │
│ Early Vote (South)            ┆ 60     

year,state,state_postal,state_fips,county_name,county_fips,precinct,party,votes
i64,str,str,i64,str,i64,str,str,i64
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0011 ALHAMBRA""","""democrat""",364
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0011 ALHAMBRA""","""republican""",140
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0057 BONSALL PARK""","""democrat""",291
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0057 BONSALL PARK""","""republican""",98
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0134 CORDOVA""","""democrat""",400
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0134 CORDOVA""","""republican""",218
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0261 GRANADA""","""democrat""",1256
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0261 GRANADA""","""republican""",366
2016,"""Arizona""","""AZ""",4,"""Maricopa County""",4013,"""0332 KEIM""","""democrat""",1111


In [9]:
# One row per precinct
df_pivot = df_cleaned.pivot(
    on="party",
    index=group_cols[:-1],  # everything except party
    values="votes",
    aggregate_function="sum"
).rename({"democrat": "votes_dem", "republican": "votes_rep"})

# Calculate Voting Share
df_pivot = df_pivot.with_columns([
    (pl.col("votes_dem") + pl.col("votes_rep")).alias("total_votes"),
    (pl.col("votes_dem") / (pl.col("votes_dem") + pl.col("votes_rep"))).alias("dem_share")
])

print(df_pivot.shape)
print(df_pivot.head(10))
print(df_pivot.select(["votes_dem", "votes_rep", "total_votes", "dem_share"]).describe())


(152639, 11)
shape: (10, 11)
┌──────┬─────────┬──────────────┬────────────┬───┬───────────┬───────────┬─────────────┬───────────┐
│ year ┆ state   ┆ state_postal ┆ state_fips ┆ … ┆ votes_dem ┆ votes_rep ┆ total_votes ┆ dem_share │
│ ---  ┆ ---     ┆ ---          ┆ ---        ┆   ┆ ---       ┆ ---       ┆ ---         ┆ ---       │
│ i64  ┆ str     ┆ str          ┆ i64        ┆   ┆ i64       ┆ i64       ┆ i64         ┆ f64       │
╞══════╪═════════╪══════════════╪════════════╪═══╪═══════════╪═══════════╪═════════════╪═══════════╡
│ 2016 ┆ Arizona ┆ AZ           ┆ 4          ┆ … ┆ 364       ┆ 140       ┆ 504         ┆ 0.722222  │
│ 2016 ┆ Arizona ┆ AZ           ┆ 4          ┆ … ┆ 291       ┆ 98        ┆ 389         ┆ 0.748072  │
│ 2016 ┆ Arizona ┆ AZ           ┆ 4          ┆ … ┆ 400       ┆ 218       ┆ 618         ┆ 0.647249  │
│ 2016 ┆ Arizona ┆ AZ           ┆ 4          ┆ … ┆ 1256      ┆ 366       ┆ 1622        ┆ 0.774353  │
│ 2016 ┆ Arizona ┆ AZ           ┆ 4          ┆ … ┆ 1111      ┆

In [10]:
df_pivot = df_pivot.filter(
    (pl.col("votes_dem") >= 0) & (pl.col("votes_rep") >= 0)
)
df_pivot = df_pivot.filter(pl.col("total_votes") > 0)

In [12]:
print(df_pivot.describe())

shape: (9, 12)
┌────────────┬──────────┬─────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ year     ┆ state   ┆ state_post ┆ … ┆ votes_dem ┆ votes_rep ┆ total_vot ┆ dem_share │
│ ---        ┆ ---      ┆ ---     ┆ al         ┆   ┆ ---       ┆ ---       ┆ es        ┆ ---       │
│ str        ┆ f64      ┆ str     ┆ ---        ┆   ┆ f64       ┆ f64       ┆ ---       ┆ f64       │
│            ┆          ┆         ┆ str        ┆   ┆           ┆           ┆ f64       ┆           │
╞════════════╪══════════╪═════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 149777.0 ┆ 149777  ┆ 149777     ┆ … ┆ 149777.0  ┆ 149777.0  ┆ 149777.0  ┆ 149777.0  │
│ null_count ┆ 0.0      ┆ 0       ┆ 0          ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0       │
│ mean       ┆ 2016.0   ┆ null    ┆ null       ┆ … ┆ 374.12667 ┆ 372.05485 ┆ 746.18153 ┆ 0.500541  │
│            ┆          ┆         ┆            ┆   ┆ 5         ┆ 5         ┆